# 🚀 rzdhop's clips — Quick Start Notebook

> **⚡ Quick Start**: If you just want to get up and running as fast as possible,
> simply run all cells from top to bottom. The only thing you need to change is
> the **YouTube URL** in the Configuration section and your **API keys** in Colab Secrets.

---

## What is this?

This notebook is a **minimal, beginner-friendly** implementation of
[rzdhop's clips](https://github.com/rzdhop/opensource-clipping-better), a fork of [NaufalRizqullah/opensource-clipping](https://github.com/NaufalRizqullah/opensource-clipping) —
an AI-powered tool that automatically transforms long-form videos into
cinematic short-form highlight clips with:

- 🎯 AI-curated highlight moments (free providers: Groq, Gemini, NVIDIA)
- 🎙️ Word-level timing from the video's own captions, or Faster-Whisper
- 📱 Smart face-tracking auto-framing for vertical (9:16) format
- 💬 Karaoke-style animated subtitles
- 🎬 Cinematic hook teaser intro with glitch transition

## How to use this notebook

| Step | Section | What to do |
|------|---------|------------|
| 1 | **Setup** | Run the setup cells to clone the repo & install dependencies |
| 2 | **Configuration** | Set your YouTube URL, API keys, and clip preferences |
| 3 | **Run** | Execute the pipeline — sit back and wait for your clips! |
| 4 | **Download** | Download the generated clips from the `outputs/` folder |

### ⚠️ Requirements

- **Runtime**: a **T4 GPU** (`Runtime > Change runtime type > T4 GPU`) only matters when Whisper
  transcribes. With the video's own captions, a CPU runtime is fine.
- **API Key**: a free [Groq](https://console.groq.com/keys) or [Gemini](https://aistudio.google.com/apikey) key -- at least one

---

## 1. Setup — Clone Repository & Install Dependencies

This section clones the rzdhop's clips repository and installs all
required Python packages. It also installs **FFmpeg** (needed for video
processing) and **Deno** (used internally by some modules).

> 💡 **Tip**: This cell only needs to run once per Colab session.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# 1a. Clone the repository into the current working directory
# ──────────────────────────────────────────────────────────────────────
!rm -rf ./* ./.*
!git clone https://github.com/rzdhop/opensource-clipping-better.git .

In [ ]:
%%capture
# ──────────────────────────────────────────────────────────────────────
# 1b. Install system dependencies (FFmpeg + Deno) and Python packages
# ──────────────────────────────────────────────────────────────────────
import os

# FFmpeg — required for video encoding/decoding
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# Deno — used by internal subtitle rendering modules
!curl -fsSL https://deno.land/install.sh | sh
os.environ["PATH"] += ":/root/.deno/bin"

# Python dependencies
!pip install -r requirements.txt

## 2. Configuration — Set Your Preferences

This is the **only section you need to edit**. Configure the following:

| Parameter | Description | Default |
|-----------|-------------|---------|
| `SOURCE_URL` | The YouTube video to clip; the download cell fetches it | *(yours)* |
| `VIDEO_FILE` | Where the download cell saves the video | `talk.mp4` |
| `TRANSCRIPT_FILE` | Local `.vtt`/`.srt`. Skips Whisper entirely. Leave empty to transcribe | `talk.en.vtt` |
| `TOTAL_CLIPS` | How many highlight clips to generate | `3` |
| `WHISPER_MODEL` | Whisper model size for transcription accuracy | `large-v3` |
| `WHISPER_DEVICE` | Device for Whisper inference (`auto`, `cuda` or `cpu`) | `auto` |
| `WHISPER_COMPUTE_TYPE` | Compute precision (`auto`, `float16`, `float32`, ...) | `auto` |

### Whisper Model Options

| Model | Speed | Accuracy | VRAM |
|-------|-------|----------|------|
| `tiny` | ⚡⚡⚡⚡⚡ | ★☆☆☆☆ | ~1 GB |
| `base` | ⚡⚡⚡⚡ | ★★☆☆☆ | ~1 GB |
| `small` | ⚡⚡⚡ | ★★★☆☆ | ~2 GB |
| `medium` | ⚡⚡ | ★★★★☆ | ~5 GB |
| `large-v3` | ⚡ | ★★★★★ | ~10 GB |

> 💡 **Recommendation**: Use `large-v3` with `float16` on Colab T4 GPU for best
> results. If you're on a free Kaggle notebook or CPU-only environment, use
> `small` or `medium` with `float32`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CONFIGURATION — Edit the values below to match your preferences
# ══════════════════════════════════════════════════════════════════════

# ── YouTube URL ──────────────────────────────────────────────────────
# Paste the YouTube video URL you want to generate clips from.
SOURCE_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"  # <-- CHANGE THIS

# ── Local files ──────────────────────────────────────────────────────
# The download cell below saves the video and its captions here. The
# pipeline itself downloads nothing: it reads these two local files.
VIDEO_FILE = "talk.mp4"
TRANSCRIPT_FILE = "talk.en.vtt"   # "" to transcribe with Whisper instead

# ── Total Clips ─────────────────────────────────────────────────────
# Number of highlight clips to generate from the video.
# More clips = longer processing time. Start small (1-3) for testing.
TOTAL_CLIPS = 3

# ── Whisper Configuration ────────────────────────────────────────────
# These control the speech-to-text transcription step.
WHISPER_MODEL = "large-v3"        # Model size (see table above)
WHISPER_DEVICE = "auto"            # "auto" picks the GPU when Whisper can use one
WHISPER_COMPUTE_TYPE = "auto"      # "auto", or "float16" for Colab T4, "float32" for CPU

# ══════════════════════════════════════════════════════════════════════
#  OPTIONAL — Advanced settings (safe to leave as defaults)
# ══════════════════════════════════════════════════════════════════════

ASPECT_RATIO = "9:16"              # Output aspect ratio: "9:16", "16:9", "1:1"
FONT_STYLE = "HORMOZI"             # Subtitle style: "DEFAULT", "HORMOZI", "CINEMATIC", "STORYTELLER"
HOOK_DURATION = 3                  # Hook teaser duration in seconds
WORDS_PER_SUBTITLE = 5             # Max words per subtitle group

print("✅ Configuration loaded!")
print(f"   URL           : {SOURCE_URL}")
print(f"   Files         : {VIDEO_FILE}, {TRANSCRIPT_FILE or '(Whisper)'}")
print(f"   Total Clips   : {TOTAL_CLIPS}")
print(f"   Whisper Model : {WHISPER_MODEL} ({WHISPER_DEVICE}, {WHISPER_COMPUTE_TYPE})")
print(f"   Aspect Ratio  : {ASPECT_RATIO}")
print(f"   Font Style    : {FONT_STYLE}")

## 3. API Key Setup

The analysis runs on a **chain of free AI providers**. It needs at least one of
**Groq** or **Gemini** — both free, no card. NVIDIA is an optional backup: it is
too slow to carry a job on its own, so a job with only an NVIDIA key is refused.

Store the keys in **Colab Secrets**:

1. Click the 🔑 **Key icon** in the left sidebar of Colab
2. Add a secret named `GROQ_API_KEY` and/or `GOOGLE_API_KEY`
3. Paste the key as the value
4. Toggle the **"Notebook access"** switch ON

> 🔗 Groq: https://console.groq.com/keys · Gemini: https://aistudio.google.com/apikey

**Optional keys** (for extra features):
- `NVIDIA_API_KEY` — A backup link in the chain ([get one here](https://build.nvidia.com/))
- `PEXELS_API_KEY` — Enables automatic B-roll stock footage ([get one here](https://www.pexels.com/api/))
- `HF_TOKEN` — Enables split-screen / camera-switch mode ([get one here](https://huggingface.co/settings/tokens))

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Load API keys from Colab Secrets and create the .env file
# ──────────────────────────────────────────────────────────────────────
from google.colab import userdata
from pathlib import Path


def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:  # SecretNotFoundError, or notebook access switched off
        return ""


# At least one of these two
GROQ_API_KEY = secret("GROQ_API_KEY")
GOOGLE_API_KEY = secret("GOOGLE_API_KEY")

# Optional — leave blank if you don't have them
NVIDIA_API_KEY = secret("NVIDIA_API_KEY")
PEXELS_API_KEY = secret("PEXELS_API_KEY")
HF_TOKEN = secret("HF_TOKEN")

# Write the .env file so the pipeline can read the keys
env_text = f"""# Auto-generated by Quick Start notebook
GROQ_API_KEY={GROQ_API_KEY}
GOOGLE_API_KEY={GOOGLE_API_KEY}
NVIDIA_API_KEY={NVIDIA_API_KEY}
PEXELS_API_KEY={PEXELS_API_KEY}
HF_TOKEN={HF_TOKEN}
"""
Path(".env").write_text(env_text, encoding="utf-8")

# Status check
print("🔑 API Key Status:")
print(f"   GROQ_API_KEY   : {'✅ Set' if GROQ_API_KEY else '⚪ Not set'}")
print(f"   GOOGLE_API_KEY : {'✅ Set' if GOOGLE_API_KEY else '⚪ Not set'}")
print(f"   NVIDIA_API_KEY : {'✅ Set' if NVIDIA_API_KEY else '⚪ Not set (optional backup)'}")
print(f"   PEXELS_API_KEY : {'✅ Set' if PEXELS_API_KEY else '⚪ Not set (optional)'}")
print(f"   HF_TOKEN       : {'✅ Set' if HF_TOKEN else '⚪ Not set (optional)'}")

# The analysis runs on a chain of free providers. It needs Groq or Gemini:
# a job whose only key is NVIDIA's is refused before it starts.
if not (GROQ_API_KEY or GOOGLE_API_KEY):
    print("\n⚠️  Neither GROQ_API_KEY nor GOOGLE_API_KEY is set -- the analysis will refuse to start.")
    print("   Groq:   https://console.groq.com/keys")
    print("   Gemini: https://aistudio.google.com/apikey")

## 4. Run the Pipeline 🎬

This cell runs the full AI clipping pipeline on your **local files**:

1. **Ingest** — Reads the local `.mp4`; nothing is downloaded here
2. **Transcript** — Parses the local `.vtt` and skips Whisper entirely. Leave `TRANSCRIPT_FILE` empty to transcribe with Faster-Whisper instead
3. **Analyze** — the free-provider chain (Groq → Gemini → NVIDIA) picks the most engaging moments
4. **Render** — Generates clips with face-tracking, subtitles, and hook teasers

> ⏱️ **Estimated time**: a few minutes with `--transcript`. Without it, add the Whisper pass — minutes on a GPU, hours on CPU.

> ⚠️ Make sure the acquisition cell above actually produced `VIDEO_FILE` and `TRANSCRIPT_FILE`.


In [ ]:
# ---- Acquire the inputs --------------------------------------------------
# This pipeline downloads nothing. Fetch the video and its subtitles with
# your own tool first; yt-dlp is just the most common choice.
#
# --write-auto-subs grabs YouTube's auto-captions, which carry per-word
# timing tags. Passing the resulting .vtt to --transcript skips Whisper
# entirely: minutes instead of hours, and no GPU required.

!yt-dlp -f "bv*[vcodec!*=av01]+ba/b" --merge-output-format mp4 \
  --write-auto-subs --sub-langs "en.*,id.*" --sub-format vtt --convert-subs vtt \
  -o "talk.%(ext)s" "{SOURCE_URL}"

!ls -la talk.*


In [ ]:
import glob
import os

# yt-dlp names captions after their language track (talk.en.vtt,
# talk.en-orig.vtt, ...). Use the configured one, else whatever it saved,
# else let Whisper transcribe.
if TRANSCRIPT_FILE and not os.path.isfile(TRANSCRIPT_FILE):
    found = sorted(glob.glob("talk.*.vtt"))
    TRANSCRIPT_FILE = found[0] if found else ""
print(f"🎬 Video     : {VIDEO_FILE}")
print(f"📝 Transcript: {TRANSCRIPT_FILE or '(none -- Whisper will transcribe)'}")

!python main.py \
  --video "{VIDEO_FILE}" \
  --transcript "{TRANSCRIPT_FILE}" \
  --source-url "{SOURCE_URL}" \
  --clips {TOTAL_CLIPS} \
  --ratio "{ASPECT_RATIO}" \
  --font-style "{FONT_STYLE}" \
  --hook-duration {HOOK_DURATION} \
  --words-per-sub {WORDS_PER_SUBTITLE} \
  --whisper-model "{WHISPER_MODEL}" \
  --whisper-device "{WHISPER_DEVICE}" \
  --whisper-compute-type "{WHISPER_COMPUTE_TYPE}"


## 5. View & Download Results

After the pipeline completes, your clips are saved in the `outputs/` folder.
Run the cells below to preview and download them.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# List all generated output files
# ──────────────────────────────────────────────────────────────────────
import os

outputs_dir = "outputs"
if os.path.exists(outputs_dir):
    files = os.listdir(outputs_dir)
    video_files = [f for f in files if f.endswith(".mp4")]
    print(f"🎬 Generated {len(video_files)} clip(s):\n")
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(outputs_dir, f)) / (1024 * 1024)
        icon = "🎥" if f.endswith(".mp4") else "🖼️" if f.endswith((".jpg", ".png")) else "📄"
        print(f"   {icon} {f} ({size_mb:.1f} MB)")
else:
    print("❌ No outputs found. Make sure the pipeline ran successfully.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Download all output files as a ZIP archive
# ──────────────────────────────────────────────────────────────────────
import shutil
from google.colab import files

if os.path.exists(outputs_dir) and os.listdir(outputs_dir):
    zip_name = "clipping_outputs"
    shutil.make_archive(zip_name, "zip", outputs_dir)
    print(f"📦 Created {zip_name}.zip — downloading now...")
    files.download(f"{zip_name}.zip")
else:
    print("❌ No outputs to download.")

---

## 📖 Troubleshooting

| Problem | Solution |
|---------|----------|
| The analysis refuses to start | Add `GROQ_API_KEY` or `GOOGLE_API_KEY` to Colab Secrets (🔑 sidebar) |
| `CUDA out of memory` | Use a smaller Whisper model (`small` or `medium`) |
| `float16 not supported` | Change `WHISPER_COMPUTE_TYPE` to `float32` |
| Pipeline takes too long | Reduce `TOTAL_CLIPS` to `1` for testing |
| No B-roll footage | Add `PEXELS_API_KEY` to Colab Secrets |

## 🔗 Links

- **GitHub**: [rzdhop/opensource-clipping-better](https://github.com/rzdhop/opensource-clipping-better), a fork of [NaufalRizqullah/opensource-clipping](https://github.com/NaufalRizqullah/opensource-clipping)
- **Full Documentation**: See the [README](https://github.com/rzdhop/opensource-clipping-better/blob/main/README.md) for all CLI options
- **Get a Groq API Key**: [console.groq.com/keys](https://console.groq.com/keys)
- **Get a Gemini API Key**: [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
- **Get Pexels API Key**: [pexels.com/api](https://www.pexels.com/api/)